# Module 3.1: UniFi Switch Enterprise 8 PoE Topology, Static IP Subnet Allocation & Tailscale Mesh Unification

## Homelab Expansion (Chapter 2) - Step 3: Multi-Node Networking & Enterprise Switch Infrastructure

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand the engineering advantages and hardware compatibility of deploying the **Ubiquiti UniFi Switch Enterprise 8 PoE (`USW-Enterprise-8-PoE`)** as your multi-node homelab core switch (8× 2.5 GbE PoE+ ports, 2× 10G SFP+ ports, 80 Gbps switching capacity, fanless passive cooling).
2. Explain **IEEE 802.3af/at Active PoE+** negotiation (why non-PoE hosts like `homelab-pi` and `homelab-gpu` are 100% safe to connect) and understand the **PoE+ (32W) vs. PoE++ (60W)** boundary that informs Chapter 3's `UNAS 2` power topology.
3. Design a contiguous static IP allocation scheme (`10.0.0.200`–`10.0.0.210`) covering `homelab-pi`, `homelab-gpu`, and upcoming Chapter 3 UniFi infrastructure (`UNAS 2` and managed switch IP).
4. Configure a DHCP MAC address reservation (`10.0.0.201`) for `homelab-gpu`, verify physical link negotiation with `ethtool`, and benchmark full wire-speed node-to-node throughput (`~940 Mbps` to `homelab-pi`) using `iperf3`.
5. Provision `homelab-gpu` onto your Tailscale WireGuard mesh (applying our Chapter 1 Avahi `deny-interfaces=tailscale0` lesson upfront) and configure unified workstation `~/.ssh/config` aliases.

## 3.1.1 Conceptual Background: Engineering a Multi-Gigabit Homelab Core with `USW-Enterprise-8-PoE`

When your homelab grows from a single Raspberry Pi into a multi-node cluster (`homelab-pi` + `homelab-gpu` + Chapter 3's `UNAS 2` NAS), plugging devices into random Wi-Fi bands or daisy-chaining consumer router ports creates latency jitter, CPU contention, and port exhaustion.

Replacing a consumer 1GbE desktop switch with the **Ubiquiti UniFi Switch Enterprise 8 PoE (`USW-Enterprise-8-PoE`)** turns your homelab shelf into a genuine multi-gigabit enterprise core—and it is **100% compatible** with both Chapter 2's nodes and Chapter 3's UniFi storage expansion.

### 1. Why the `USW-Enterprise-8-PoE` Is a Massive Upgrade for Your Lab
- **80 Gbps Non-Blocking Multi-Gig Fabric:** The `USW-Enterprise-8-PoE` provides **8× 2.5 GbE RJ45 ports** (auto-negotiating 2.5G / 1G / 100M / 10M) plus **2× 10 Gbps SFP+ cages**, backed by **80 Gbps switching capacity**, **40 Gbps total non-blocking throughput**, and a **59.52 Mpps forwarding rate**. Every health probe from `homelab-pi` (`10.0.0.200`), every media/backup transfer between `homelab-gpu` (`10.0.0.201`) and Chapter 3's `UNAS 2` (`10.0.0.202`), and every DNS query travels across dedicated switching silicon at `<0.3 ms` latency without ever touching your home router's CPU.
- **Works Plug-and-Play Out of the Box (Layer-2 Switching Before Adoption):** Even though the `USW-Enterprise-8-PoE` is a Layer-3 managed UniFi switch, you do **not** need a UniFi Cloud Gateway or controller running yet to complete Chapter 2! Out of the box, it forwards Ethernet frames at full wire speed across all ports. In **Chapter 3, Step 2**, we will deploy a self-hosted **UniFi Network Application** in Docker on `homelab-gpu` to adopt the switch and unlock live port telemetry, per-port PoE controls, and VLANs.
- **Active IEEE 802.3af/at PoE+ Safety (120W Budget, Max 32W/Port):** All 8 RJ45 ports support **active** PoE/PoE+. Before injecting 48V DC onto any copper pair, the switch performs an IEEE 802.3af/at resistance signature handshake (`25 kΩ` detection). Non-PoE devices (`homelab-pi`, `homelab-gpu`, home router) fail the PoE signature check, so the port delivers **data only**—making direct connection completely safe. (Bonus: You can even power a Raspberry Pi 4 via an optional 802.3at PoE+ HAT or USB-C PoE splitter later!)
- **Crucial Chapter 3 Preview — PoE+ vs. PoE++ Limit:** Each port on the `USW-Enterprise-8-PoE` tops out at **PoE+ (802.3at, 32W max)**. High-draw **PoE++ (802.3bt, 60W)** appliances—specifically the **UniFi UNAS 2** arriving in Chapter 3—cannot be powered directly by this switch and will instead connect through their **included 60W PoE++ injector** plugged inline on Port 4 (preserving full **2.5 GbE** data speed).
- **Silent Fanless Passive Cooling:** The `USW-Enterprise-8-PoE` uses an internal 150W AC power supply and a completely **fanless** aluminum heatsink chassis. It is dead silent, which makes it ideal for a bedroom or office homelab—but it naturally runs warm to the touch under multi-gigabit load. **Always leave at least 2–3 inches of vertical airflow clearance above its top ridged surface** (never stack the ROG Zephyrus laptop or Raspberry Pi directly on top of the switch!).

### 2. Multi-Node Port Allocation & Static IP Plan (`10.0.0.200`–`10.0.0.210`)

```mermaid
flowchart TB
    Router["Home Router / Gateway<br>10.0.0.1 (DHCP Server)"] ===|"Port 1 (Uplink)"| Switch["UniFi USW-Enterprise-8-PoE<br>8x 2.5GbE PoE+ | 2x 10G SFP+<br>80 Gbps Switching | 1.3in LCM"]
    
    Switch ===|"Port 2 (1 GbE)"| Pi["Node 1: homelab-pi<br>LAN: 10.0.0.200<br>Role: Tier-0 Pi-hole + Tailscale + Kuma"]
    Switch ===|"Port 3 (1 GbE / 2.5 GbE)"| GPU["Node 2: homelab-gpu (ROG Zephyrus)<br>LAN: 10.0.0.201<br>Role: Full Immich (CUDA/NVENC) + Ch3 Apps"]
    Switch -.-|"Port 4 (2.5 GbE -> Ch3 PoE++ Injector)"| NAS["Chapter 3: UniFi UNAS 2<br>Planned IP: 10.0.0.202<br>2x 4TB RAID 1 Storage"]
    Switch -.-|"Ports 5-8 (2.5GbE PoE+) & 9-10 (10G SFP+)"| Future["Future Expansion (.204 - .210)<br>PoE+ AP / Proxmox / 10G Fiber"]
```

| Switch Port | Link Speed Capability | Assigned Device | Static LAN IP | Notes |
| :---: | :---: | :--- | :---: | :--- |
| **Port 1** | 2.5 GbE PoE+ | Home Router / Gateway Uplink | `10.0.0.1` | Single uplink into household LAN/WAN |
| **Port 2** | 2.5 GbE PoE+ *(neg. 1G)* | **Node 1:** `homelab-pi` (Raspberry Pi 4) | `10.0.0.200` | Native 1 GbE NIC (`~940 Mbps`) |
| **Port 3** | 2.5 GbE PoE+ *(neg. 1G or 2.5G)* | **Node 2:** `homelab-gpu` (ROG Zephyrus) | `10.0.0.201` | Native 1 GbE (or **2.5 GbE** with USB-C `RTL8156` dongle) |
| **Port 4** | 2.5 GbE PoE+ *(neg. 2.5G)* | *Reserved for Chapter 3:* `UNAS 2` (via 60W PoE++ Injector) | `10.0.0.202` | Full **2.5 GbE** NAS link in Chapter 3 |
| **Mgmt** | Internal L2/L3 | `USW-Enterprise-8-PoE` Management IP | `10.0.0.203` | Adopted into UniFi Network Server in Chapter 3 |
| **Ports 5–8** | 2.5 GbE PoE+ | Future Homelab Nodes / PoE+ APs | `10.0.0.204+` | 120W shared PoE+ budget available |
| **Ports 9–10** | 10 Gbps SFP+ | Future 10GbE Server / DAC / Fiber Uplink | — | Supports 1G/10G SFP+ optics or passive DAC cables |

## 📖 Official Hardware Reference Figure

<br>

<img src="https://github.com/rd908213/Homelab-Development/blob/main/assets/hardware/unifi_usw_enterprise_8_poe.png?raw=true" width="75%">

*Reference Figure 1: Ubiquiti UniFi Switch Enterprise 8 PoE (`USW-Enterprise-8-PoE`) — showing the 1.3" color LCM touchscreen (left), 8× 2.5 GbE PoE+ RJ45 ports (center), and 2× 10G SFP+ cages (right).*

<br>

### Networking Hardware Kit Summary

| Component | Model | Technical Role |
| :--- | :--- | :--- |
| **Core Multi-Gig Switch** | **Ubiquiti `USW-Enterprise-8-PoE`** | 8× 2.5 GbE PoE+ (120W total, 32W/port), 2× 10G SFP+, 40 Gbps non-blocking throughput, silent fanless chassis, internal 150W AC PSU. |
| **USB 3.0 / USB-C Ethernet Adapter** *(Optional / If Needed)* | **Realtek `RTL8156` (2.5GbE) or `RTL8153` (1GbE)** | Required if your 2019 ROG Zephyrus (`GA502`/`GU502`) lacks a built-in RJ45 jack—or an awesome optional upgrade on `GX502` to unlock full **2.5 Gbps** throughput between `homelab-gpu` and Chapter 3's `UNAS 2`! |
| **Patch Cables** | **UniFi Patch Cables (`UACC-CABLE-PATCH-0.4M-BL-10`) or Cat 6** | Ultra-thin 0.4m bendable-boot cables rated up to 10 GbE & PoE+++ (full structured cabling covered in Chapter 3, Module 1.2). |

## 3.1.2 Step-by-Step Laboratory Guide

### Task 3.1.1: Power, Place & Wire the `USW-Enterprise-8-PoE` and Reserve `10.0.0.201`
1. **Thermal Placement & UPS Power:** Place the `USW-Enterprise-8-PoE` flat on your homelab shelf (or wall-mounted via its rear keyhole bracket) with unobstructed airspace above its top heatsink surface. Plug its AC power cord directly into a **Battery + Surge** outlet on your UPS so your entire homelab fabric stays online during power blips.
2. Wait ~30–45 seconds for the front **1.3" LCM touchscreen** to finish booting and display the UniFi status readout.
3. Wire your initial Chapter 2 links:
   - **Port 1:** Patch cable to a LAN port on your home router (`10.0.0.1`).
   - **Port 2:** Patch cable to `homelab-pi` (`10.0.0.200`).
   - **Port 3:** Patch cable to `homelab-gpu` (`10.0.0.201`).
   - *(Notice the port LEDs light up automatically—and tap the front LCM screen to confirm active link speeds on Ports 1–3!)*
4. On `homelab-gpu`, inspect your active wired Ethernet interface name, MAC (`link/ether`) address, and negotiated link speed:
   ```bash
   ip -br link
   sudo apt update && sudo apt install -y ethtool
   # Replace eth0 / enp3s0 with your wired interface name from `ip -br link`
   sudo ethtool $(ip route | awk '/default/ {print $5; exit}') | grep -E "Speed:|Duplex:|Link detected:"
   ```
   *(Expected output: `Speed: 1000Mb/s` on built-in Gigabit NICs, or `Speed: 2500Mb/s` if using a USB 3.2 `RTL8156` 2.5GbE adapter; `Duplex: Full`; `Link detected: yes`.)*
5. Open your home router's administration portal (just as you did in Chapter 1, Module 2.1 for `homelab-pi` at `10.0.0.200`), navigate to **DHCP Static Lease / Address Reservation**, and bind:
   - `homelab-gpu`'s Ethernet MAC address $\rightarrow$ **`10.0.0.201`**
   - *(Optional proactive reservation for Chapter 3)* `USW-Enterprise-8-PoE`'s MAC address (printed on the underside label and visible in your router's DHCP client list) $\rightarrow$ **`10.0.0.203`**
6. Renew the DHCP lease on `homelab-gpu` and verify it holds `10.0.0.201`:
   ```bash
   sudo netplan apply
   ip -4 -br addr show
   ```

### Task 3.1.2: Benchmark Node-to-Node Throughput Across `USW-Enterprise-8-PoE` (`iperf3`)
Let's prove that your `USW-Enterprise-8-PoE` passes traffic directly between `homelab-pi` and `homelab-gpu` at full wire speed (`~940 Mbps`, bounded strictly by the Raspberry Pi 4's 1 GbE NIC):

1. **On Node 1 (`homelab-pi`):** Install `iperf3` and start it in one-shot server listener mode:
   ```bash
   sudo apt update && sudo apt install -y iperf3
   iperf3 -s -1
   ```
2. **On Node 2 (`homelab-gpu`):** Install `iperf3` and run a 5-second bandwidth test against `homelab-pi` (`10.0.0.200`):
   ```bash
   sudo apt install -y iperf3
   iperf3 -c 10.0.0.200 -t 5
   ping -c 4 10.0.0.200
   ```
   You should see **`~935 - 945 Mbits/sec`** sustained transfer speed and sub-millisecond ping latency (`<0.4 ms`) across the `USW-Enterprise-8-PoE` ASIC backplane!

### 🖼️ Task 3.1.2 Verification: `USW-Enterprise-8-PoE` Wiring & `iperf3` Benchmark

**Verification Objective:** Capture a screenshot showing the `iperf3 -c 10.0.0.200 -t 5` result on `homelab-gpu` achieving ~940 Mbps across the `USW-Enterprise-8-PoE` switch (optionally paired with a photo of the switch LCM screen showing active links on Ports 1–3).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 3.1.3: Join `homelab-gpu` to Tailscale & Apply the Chapter 1 Avahi Fix
Register `homelab-gpu` on your Tailscale WireGuard mesh so you can manage both nodes remotely, and immediately apply our Chapter 1 lesson to prevent `avahi-daemon` from renaming the host on `tailscale0`:

```bash
# 1. Install Tailscale via official script on homelab-gpu
curl -fsSL https://tailscale.com/install.sh | sh

# 2. Authenticate and join your Tailnet
sudo tailscale up

# 3. Apply Chapter 1 Learning: Prevent Avahi mDNS hostname collisions on tailscale0
if [ -f /etc/avahi/avahi-daemon.conf ]; then
    sudo sed -i 's/#deny-interfaces=eth1/deny-interfaces=tailscale0/' /etc/avahi/avahi-daemon.conf
    sudo systemctl restart avahi-daemon
fi

# 4. Verify both homelab-pi and homelab-gpu appear in your Tailnet
tailscale status
```

### Task 3.1.4: Unified Workstation `~/.ssh/config` & Direct Node-to-Node SSH
Update `~/.ssh/config` on your workstation so you can jump into either node seamlessly using its static IP:

```text
Host homelab-pi
    HostName 10.0.0.200
    User pi
    IdentityFile ~/.ssh/id_ed25519_homelab
    IdentitiesOnly yes
    AddKeysToAgent yes

Host homelab-gpu
    HostName 10.0.0.201
    User pi
    IdentityFile ~/.ssh/id_ed25519_homelab
    IdentitiesOnly yes
    AddKeysToAgent yes
```
Verify from your workstation that both `ssh homelab-pi` and `ssh homelab-gpu` connect instantaneously without a password prompt!

### 🖼️ Task 3.1.4 Verification: Unified Tailscale Mesh & Dual-Node SSH Status

**Verification Objective:** Capture a screenshot showing `tailscale status` with both `homelab-pi` and `homelab-gpu` online, along with successful `ssh homelab-gpu` connectivity.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] `USW-Enterprise-8-PoE` powered from the UPS with proper top-surface passive cooling clearance.
- [ ] Router uplink (Port 1), `homelab-pi` (`10.0.0.200`, Port 2), and `homelab-gpu` (`10.0.0.201`, Port 3) wired with static DHCP reservations.
- [ ] Full wire-speed throughput (`~940 Mbps` to `homelab-pi`) and `<0.4 ms` latency verified via `iperf3` and `ethtool`.
- [ ] `homelab-gpu` joined to Tailscale and `tailscale0` excluded from `avahi-daemon`.
- [ ] Workstation `~/.ssh/config` updated with both `homelab-pi` and `homelab-gpu` host definitions.
- [ ] Verification screenshots captured and attached above.

Proceed to [Module 4.1: NVIDIA Drivers, Container Toolkit & Docker Runtime](../04_gpu_compute_and_workload_migration/04_1_nvidia_drivers_and_container_toolkit.ipynb).